In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from delta.tables import DeltaTable

# No removeAll(): job parameters must be respected
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"])
dbutils.widgets.text("start_date", "2011-01-01")
dbutils.widgets.text("end_date", "2026-09-30")
dbutils.widgets.text("batch_id", "")

load_type  = dbutils.widgets.get("load_type").strip().lower()
start_date = dbutils.widgets.get("start_date").strip()
end_date   = dbutils.widgets.get("end_date").strip()
batch_id   = dbutils.widgets.get("batch_id").strip()

silver_table = "silver.world_bank_food_prices"
audit_table  = "audit.pipeline_runs"
source_name  = "World Bank Real-Time Food Prices"
run_id       = str(uuid.uuid4())

def now():
    return datetime.now(timezone.utc).replace(tzinfo=None)

pipeline_start = now()
records_read = records_inserted = records_updated = records_rejected = 0
step_metrics = []

for s in ["gold", "audit", "rejects"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")

AUDIT_SCHEMA = StructType([
    StructField("run_id", StringType(), False), StructField("source_name", StringType(), True),
    StructField("layer", StringType(), True), StructField("load_type", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True), StructField("end_time", TimestampType(), True),
    StructField("records_read", LongType(), True), StructField("records_inserted", LongType(), True),
    StructField("records_updated", LongType(), True), StructField("records_rejected", LongType(), True),
    StructField("status", StringType(), True), StructField("error_message", StringType(), True),
])

def write_audit(status, error_message=None):
    end = now()
    param = (f"load_type={load_type}; start_date={start_date}; end_date={end_date}; "
             f"batch_id={batch_id or '(none)'}; duration_s={(end - pipeline_start).total_seconds():.1f}")
    row = spark.createDataFrame([(run_id, source_name, "Gold", load_type, param, pipeline_start, end,
            int(records_read), int(records_inserted), int(records_updated), int(records_rejected),
            status, str(error_message)[:4000] if error_message else None)], schema=AUDIT_SCHEMA)
    row.select(*spark.table(audit_table).columns).write.format("delta").mode("append").saveAsTable(audit_table)

def log_step(table, rows_read, ins, upd, rej, t0):
    t1 = now()
    step_metrics.append((run_id, table, int(rows_read), int(ins), int(upd), int(rej), t0, t1, (t1 - t0).total_seconds()))
    print(f"{table}: read={rows_read} inserted={ins} updated={upd} rejected={rej} ({(t1-t0).total_seconds():.1f}s)")

def merge_into(target, src_df, keys):
    """Conditional MERGE: update only when a business value changed. Returns (inserted, updated)."""
    non_keys = [c for c in src_df.columns if c not in keys]
    tracked  = [c for c in non_keys if c not in ("gold_loaded_at", "_gold_run_id", "_silver_run_id")]
    cond     = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    changed  = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in tracked)
    (DeltaTable.forName(spark, target).alias("t")
        .merge(src_df.alias("s"), cond)
        .whenMatchedUpdate(condition=changed, set={c: f"s.`{c}`" for c in non_keys})
        .whenNotMatchedInsertAll()
        .execute())
    m = DeltaTable.forName(spark, target).history(1).select("operationMetrics").first()[0] or {}
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))

print("Gold run:", run_id, "| load_type:", load_type)

In [0]:
try:
    spark.sql("""CREATE TABLE IF NOT EXISTS gold.dim_date (
        date_key INT, full_date DATE, year INT, quarter INT, month INT,
        month_name STRING, year_month STRING) USING DELTA""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.dim_location (
        location_key BIGINT, ISO3 STRING, geo_id STRING, country STRING, adm1_name STRING,
        adm2_name STRING, mkt_name STRING, lat DOUBLE, lon DOUBLE, currency STRING,
        gold_loaded_at TIMESTAMP) USING DELTA CLUSTER BY (location_key)""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.dim_commodity (
        commodity_key BIGINT, commodity_name STRING, commodity_group STRING,
        is_fao_series BOOLEAN, gold_loaded_at TIMESTAMP) USING DELTA""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.fact_food_price (
        location_key BIGINT, date_key INT, commodity_key BIGINT,
        price DOUBLE, open_price DOUBLE, high_price DOUBLE, low_price DOUBLE, close_price DOUBLE,
        inflation DOUBLE, trust_score DOUBLE,
        _silver_run_id STRING, _gold_run_id STRING, gold_loaded_at TIMESTAMP)
        USING DELTA CLUSTER BY (date_key, location_key, commodity_key)""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.fact_food_price_index (
        location_key BIGINT, date_key INT,
        open_index DOUBLE, high_index DOUBLE, low_index DOUBLE, close_index DOUBLE,
        inflation DOUBLE, trust_score DOUBLE, data_coverage DOUBLE, data_coverage_recent DOUBLE,
        index_confidence_score DOUBLE, spatially_interpolated BOOLEAN,
        _silver_run_id STRING, _gold_run_id STRING, gold_loaded_at TIMESTAMP)
        USING DELTA CLUSTER BY (date_key, location_key)""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.etl_watermark (
        pipeline STRING, last_silver_processed_at TIMESTAMP, updated_at TIMESTAMP) USING DELTA""")

    spark.sql("""CREATE TABLE IF NOT EXISTS gold.pipeline_step_metrics (
        run_id STRING, table_name STRING, rows_read BIGINT, rows_inserted BIGINT, rows_updated BIGINT,
        rows_rejected BIGINT, start_time TIMESTAMP, end_time TIMESTAMP, duration_seconds DOUBLE) USING DELTA""")
    print("Gold tables ready.")
except Exception as exc:
    write_audit("FAILURE", exc); raise

In [0]:
try:
    silver = spark.table(silver_table)
    cols = set(silver.columns)

    if load_type == "full":
        src = silver
    elif load_type == "backfill":
        src = silver.filter((F.col("price_date") >= start_date) & (F.col("price_date") <= end_date))
    else:  # incremental: only rows Silver inserted/changed since last successful Gold run
        wm = spark.sql("SELECT max(last_silver_processed_at) FROM gold.etl_watermark "
                       "WHERE pipeline = 'gold_food_prices'").first()[0]
        src = silver if wm is None else silver.filter(F.col("_silver_processed_at") > F.lit(wm))

    if batch_id:
        src = src.filter(F.col("_silver_run_id") == batch_id)

    # No .cache(): not supported on serverless. One aggregate gives both values.
    stats = src.agg(F.count("*").alias("n"), F.max("_silver_processed_at").alias("wm")).first()
    records_read = stats["n"]
    new_watermark = stats["wm"]
    print("Silver rows selected:", records_read)

    # Commodities = base columns that have the full o_/h_/l_/c_/inflation_/trust_ family
    commodities = sorted(c for c in cols
        if f"o_{c}" in cols and f"trust_{c}" in cols
        and not c.startswith(("o_", "h_", "l_", "c_", "inflation_", "trust_")))
    print("Commodities detected:", len(commodities))
except Exception as exc:
    write_audit("FAILURE", exc); raise

if records_read == 0:
    write_audit("SUCCESS")
    dbutils.notebook.exit("No new Silver rows; Gold unchanged.")

In [0]:
try:
    # ---- dim_date (static calendar, built once) ----
    t0 = now()
    if spark.table("gold.dim_date").limit(1).count() == 0:
        (spark.sql("SELECT explode(sequence(DATE'2005-01-01', DATE'2035-12-31', INTERVAL 1 DAY)) AS full_date")
            .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
                    F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"),
                    F.month("full_date").alias("month"), F.date_format("full_date", "MMMM").alias("month_name"),
                    F.date_format("full_date", "yyyy-MM").alias("year_month"))
            .write.format("delta").mode("append").saveAsTable("gold.dim_date"))
        log_step("gold.dim_date", 0, spark.table("gold.dim_date").count(), 0, 0, t0)

    # ---- dim_location (SCD type 1: latest attributes win) ----
    t0 = now()
    loc = (src.groupBy("ISO3", "geo_id")
              .agg(*[F.max_by(c, "price_date").alias(c)
                     for c in ["country", "adm1_name", "adm2_name", "mkt_name", "lat", "lon", "currency"]])
              .withColumn("location_key", F.xxhash64("ISO3", "geo_id"))
              .withColumn("gold_loaded_at", F.current_timestamp())
              .select("location_key", "ISO3", "geo_id", "country", "adm1_name", "adm2_name",
                      "mkt_name", "lat", "lon", "currency", "gold_loaded_at"))
    n = loc.count()
    ins, upd = merge_into("gold.dim_location", loc, ["location_key"])
    log_step("gold.dim_location", n, ins, upd, 0, t0)
    records_inserted += ins; records_updated += upd

    # ---- dim_commodity ----
    def commodity_group(name):
        n = name.lower()
        rules = [("cereal", ["rice","wheat","maize","millet","sorghum","fonio","teff","couscous","pasta","bread"]),
                 ("tuber_root", ["cassava","potato","yam","cocoyam","gari","plantain"]),
                 ("pulse_legume", ["bean","lentil","chickpea","cowpea","pulse","groundnut"]),
                 ("meat_livestock", ["meat","livestock"]),
                 ("fish_seafood", ["fish","shrimp"]),
                 ("dairy_egg", ["milk","cheese","butter","yogurt","egg"]),
                 ("fruit", ["apple","banana","mango","orange","lemon","date","watermelon"]),
                 ("vegetable", ["cabbage","carrot","chili","cucumber","eggplant","garlic","lettuce",
                                "okra","onion","pepper","pumpkin","tomato"]),
                 ("oil_sugar_other", ["oil","sugar","salt","honey","tea","coffee","cocoa","sesame"])]
        for g, kws in rules:
            if any(k in n for k in kws): return g
        return "other"
    group_udf = F.udf(commodity_group, StringType())

    t0 = now()
    com = (spark.createDataFrame([(c,) for c in commodities], "commodity_name string")
              .withColumn("commodity_key", F.xxhash64("commodity_name"))
              .withColumn("commodity_group", group_udf("commodity_name"))
              .withColumn("is_fao_series", F.col("commodity_name").endswith("_fao"))
              .withColumn("gold_loaded_at", F.current_timestamp())
              .select("commodity_key", "commodity_name", "commodity_group", "is_fao_series", "gold_loaded_at"))
    ins, upd = merge_into("gold.dim_commodity", com, ["commodity_key"])
    log_step("gold.dim_commodity", len(commodities), ins, upd, 0, t0)
    records_inserted += ins; records_updated += upd
except Exception as exc:
    write_audit("FAILURE", exc); raise

In [0]:
try:
    # ---- fact_food_price: unpivot ~130 commodity column families into rows ----
    t0 = now()
    parts = [f"'{c}', `{c}`, `o_{c}`, `h_{c}`, `l_{c}`, `c_{c}`, `inflation_{c}`, `trust_{c}`" for c in commodities]
    stack_expr = (f"stack({len(commodities)}, {', '.join(parts)}) AS "
                  "(commodity_name, price, open_price, high_price, low_price, close_price, inflation, trust_score)")
    needed = (["ISO3", "geo_id", "price_date", "_silver_run_id"] +
              [x for c in commodities for x in (c, f"o_{c}", f"h_{c}", f"l_{c}", f"c_{c}", f"inflation_{c}", f"trust_{c}")])

    fact = (src.select(*needed)
               .select("ISO3", "geo_id", "price_date", "_silver_run_id", F.expr(stack_expr))
               .filter(F.col("price").isNotNull() | F.col("close_price").isNotNull())
               .withColumn("location_key", F.xxhash64("ISO3", "geo_id"))
               .withColumn("date_key", F.date_format("price_date", "yyyyMMdd").cast("int"))
               .withColumn("commodity_key", F.xxhash64("commodity_name"))
               .withColumn("_gold_run_id", F.lit(run_id))
               .withColumn("gold_loaded_at", F.current_timestamp())
               .select("location_key", "date_key", "commodity_key", "price", "open_price", "high_price",
                       "low_price", "close_price", "inflation", "trust_score",
                       "_silver_run_id", "_gold_run_id", "gold_loaded_at"))

    # Data-quality rule: negative price values are quarantined
    price_columns = [
        "price", "open_price", "high_price",
        "low_price", "close_price"
    ]

    is_bad = F.lit(False)

    for c in price_columns:
        is_bad = is_bad | (F.col(c) < 0)

    bad_df = fact.filter(is_bad)
    good_df = fact.filter(~is_bad)

    n_rej = bad_df.count()

    if n_rej:
        (bad_df.withColumn("reject_reason", F.lit("Negative price"))
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable("rejects.gold_fact_rejects"))

    # Stage once so the MERGE doesn't recompute the wide unpivot
    (good_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("gold.fact_food_price_staging"))

    staged = spark.table("gold.fact_food_price_staging")
    n = staged.count()

    ins, upd = merge_into(
        "gold.fact_food_price",
        staged,
        ["location_key", "date_key", "commodity_key"]
    )

    spark.sql("DROP TABLE IF EXISTS gold.fact_food_price_staging")

    log_step("gold.fact_food_price", n, ins, upd, n_rej, t0)
    records_inserted += ins
    records_updated += upd
    records_rejected += n_rej
    # Stage once so the MERGE doesn't recompute the wide unpivot
    (good_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable("gold.fact_food_price_staging"))
    staged = spark.table("gold.fact_food_price_staging")
    n = staged.count()
    ins, upd = merge_into("gold.fact_food_price", staged, ["location_key", "date_key", "commodity_key"])
    spark.sql("DROP TABLE IF EXISTS gold.fact_food_price_staging")
    log_step("gold.fact_food_price", n, ins, upd, n_rej, t0)
    records_inserted += ins; records_updated += upd; records_rejected += n_rej

    # ---- fact_food_price_index ----
    t0 = now()
    idx = (src.filter(F.col("c_food_price_index").isNotNull() | F.col("o_food_price_index").isNotNull())
              .select(F.xxhash64("ISO3", "geo_id").alias("location_key"),
                      F.date_format("price_date", "yyyyMMdd").cast("int").alias("date_key"),
                      F.col("o_food_price_index").alias("open_index"),
                      F.col("h_food_price_index").alias("high_index"),
                      F.col("l_food_price_index").alias("low_index"),
                      F.col("c_food_price_index").alias("close_index"),
                      F.col("inflation_food_price_index").alias("inflation"),
                      F.col("trust_food_price_index").alias("trust_score"),
                      "data_coverage", "data_coverage_recent", "index_confidence_score",
                      "spatially_interpolated", "_silver_run_id",
                      F.lit(run_id).alias("_gold_run_id"), F.current_timestamp().alias("gold_loaded_at")))
    n = idx.count()
    ins, upd = merge_into("gold.fact_food_price_index", idx, ["location_key", "date_key"])
    log_step("gold.fact_food_price_index", n, ins, upd, 0, t0)
    records_inserted += ins; records_updated += upd
except Exception as exc:
    write_audit("FAILURE", exc); raise

In [0]:
try:
    # Referential integrity: fail instead of publishing orphans
    orphans = spark.sql("""
        SELECT
          (SELECT count(*) FROM gold.fact_food_price f LEFT ANTI JOIN gold.dim_location d USING (location_key)) +
          (SELECT count(*) FROM gold.fact_food_price f LEFT ANTI JOIN gold.dim_commodity d USING (commodity_key)) +
          (SELECT count(*) FROM gold.fact_food_price f LEFT ANTI JOIN gold.dim_date d USING (date_key)) +
          (SELECT count(*) FROM gold.fact_food_price_index f LEFT ANTI JOIN gold.dim_location d USING (location_key))
          AS orphans""").first()[0]
    if orphans:
        raise ValueError(f"Referential integrity failed: {orphans} orphan fact rows")

    # Read optimisation (applies the Liquid Clustering layout and compacts files)
    for t in ["gold.fact_food_price", "gold.fact_food_price_index", "gold.dim_location"]:
        spark.sql(f"OPTIMIZE {t}")

    # Persist per-table metrics and the watermark only after everything succeeded
    (spark.createDataFrame(step_metrics,
        "run_id string, table_name string, rows_read long, rows_inserted long, rows_updated long, "
        "rows_rejected long, start_time timestamp, end_time timestamp, duration_seconds double")
        .write.format("delta").mode("append").saveAsTable("gold.pipeline_step_metrics"))
    (spark.createDataFrame([("gold_food_prices", new_watermark, now())],
        "pipeline string, last_silver_processed_at timestamp, updated_at timestamp")
        .write.format("delta").mode("append").saveAsTable("gold.etl_watermark"))

    write_audit("SUCCESS")
    print("Gold run complete.")
except Exception as exc:
    write_audit("FAILURE", exc); raise

In [0]:
spark.sql("""CREATE OR REPLACE VIEW gold.vw_pipeline_runs AS
  SELECT run_id, layer, load_type, status, start_time, end_time,
         timestampdiff(SECOND, start_time, end_time) AS duration_seconds,
         records_read, records_inserted, records_updated, records_rejected, error_message
  FROM audit.pipeline_runs""")

spark.sql("""CREATE OR REPLACE VIEW gold.vw_silver_rejects_by_reason AS
  SELECT reject_reason, ISO3, country, count(*) AS rejected_rows
  FROM rejects.silver_food_price_rejects_v2 GROUP BY reject_reason, ISO3, country""")

spark.sql("CREATE OR REPLACE VIEW gold.vw_gold_step_metrics AS SELECT * FROM gold.pipeline_step_metrics")
print("Views created.")